In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_rate=0.1, use_dropout=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))
        
        layers.extend([
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ])
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))
        
        self.double_conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.double_conv(x)

In [3]:
class ResNet50UNet(nn.Module):
    """
    U-Net with frozen ResNet50 pretrained encoder.
    - Classification head at bottleneck for damage type (6 classes)
    - Segmentation decoder for defect mask
    """
    def __init__(self, in_channels=3, num_classes=6, dropout_rate=0.15):
        super().__init__()
        
        # Load pretrained ResNet50
        resnet50 = models.resnet50(pretrained=True)
        
        # Extract encoder layers (before avgpool and fc)
        self.conv1 = resnet50.conv1
        self.bn1 = resnet50.bn1
        self.relu = resnet50.relu
        self.maxpool = resnet50.maxpool
        self.layer1 = resnet50.layer1  # 256 channels
        self.layer2 = resnet50.layer2  # 512 channels
        self.layer3 = resnet50.layer3  # 1024 channels
        self.layer4 = resnet50.layer4  # 2048 channels
        
        # Freeze all ResNet50 weights
        for param in self.parameters():
            param.requires_grad = False
        
        # Classification head from bottleneck (2048 channels at 8x8)
        self.avg_pool_cls = nn.AdaptiveAvgPool2d(1)
        self.class_head = nn.Sequential(
            nn.Linear(2048, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        for param in self.class_head.parameters():
            param.requires_grad = True
        
        # Decoder bottleneck
        self.bottleneck = DoubleConv(2048, 1024, dropout_rate=0.2, use_dropout=True)
        
        # Reduce skip-connection channels to keep the decoder compact
        self.skip4 = nn.Conv2d(1024, 512, kernel_size=1)
        self.skip3 = nn.Conv2d(512, 256, kernel_size=1)
        self.skip2 = nn.Conv2d(256, 128, kernel_size=1)
        self.skip1 = nn.Conv2d(64, 64, kernel_size=1)
        
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512 + 512, 512, dropout_rate, use_dropout=True)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256 + 256, 256, dropout_rate, use_dropout=True)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128 + 128, 128, dropout_rate, use_dropout=True)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64 + 64, 64, dropout_rate, use_dropout=True)
        
        self.up0 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)
        
        # Enable gradients for decoder and new heads
        for module in [self.bottleneck, self.skip4, self.skip3, self.skip2, self.skip1,
                       self.up4, self.up3, self.up2, self.up1, self.up0,
                       self.dec1, self.dec2, self.dec3, self.dec4, self.final_conv]:
            for param in module.parameters():
                param.requires_grad = True

    def forward(self, x):
        # Encoder (frozen)
        e0 = self.relu(self.bn1(self.conv1(x)))  # 64, 128x128
        e0p = self.maxpool(e0)                   # 64, 64x64
        
        e1 = self.layer1(e0p)   # 256, 64x64
        e2 = self.layer2(e1)    # 512, 32x32
        e3 = self.layer3(e2)    # 1024, 16x16
        e4 = self.layer4(e3)    # 2048, 8x8
        
        # Bottleneck
        b = self.bottleneck(e4)  # 1024, 8x8
        
        # Classification head
        class_logits = self.avg_pool_cls(e4)  # 2048, 1, 1
        class_logits = class_logits.view(class_logits.size(0), -1)
        class_logits = self.class_head(class_logits)  # (batch, num_classes)
        
        # Segmentation decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, self.skip4(e3)], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        d3 = torch.cat([d3, self.skip3(e2)], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, self.skip2(e1)], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, self.skip1(e0)], dim=1)
        d1 = self.dec1(d1)
        
        d0 = self.up0(d1)
        seg_logits = self.final_conv(d0)
        
        return seg_logits, class_logits

In [4]:
class MagneticTilesDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, img_size=(256, 256)):
        self.transform = transform
        self.img_size  = img_size
        self.pairs     = []
        self.class_to_idx = {'MT_Blowhole': 0, 'MT_Break': 1, 'MT_Crack': 2, 'MT_Fray': 3, 'MT_Free': 4, 'MT_Uneven': 5}

        classes = ['MT_Blowhole', 'MT_Break', 'MT_Crack', 'MT_Fray', 'MT_Free', 'MT_Uneven']

        # Structure: root_dir/split/MT_*/Imgs/*.jpg  &  root_dir/split/MT_*/GTs/*.png
        for cls in classes:
            imgs_dir = os.path.join(root_dir, split, cls, 'Imgs')
            gts_dir  = os.path.join(root_dir, split, cls, 'GTs')
            if not os.path.isdir(imgs_dir) or not os.path.isdir(gts_dir):
                print(f"Warning: {imgs_dir} or {gts_dir} not found, skipping.")
                continue
            gt_files  = set(os.listdir(gts_dir))
            jpg_files = sorted([f for f in os.listdir(imgs_dir) if f.endswith('.jpg')])
            for jpg in jpg_files:
                base = os.path.splitext(jpg)[0]
                png  = base + '.png'
                if png in gt_files:
                    self.pairs.append((
                        os.path.join(imgs_dir, jpg),
                        os.path.join(gts_dir,  png),
                        self.class_to_idx[cls]
                    ))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, gt_path, cls_idx = self.pairs[idx]

        img   = Image.open(img_path).convert('RGB')
        label = Image.open(gt_path).convert('L')

        img   = img.resize(self.img_size)
        label = label.resize(self.img_size)

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        label = transforms.ToTensor()(label)
        label = (label > 0.5).float()

        return img, label, cls_idx

In [5]:
class EarlyStopping:
    def __init__(self, patience=15, min_delta=1e-4, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif self.best_loss - val_loss > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model)
        else:
            self.counter += 1
            
        if self.counter >= self.patience:
            if self.restore_best_weights:
                model.load_state_dict(self.best_weights)
            return True
        return False
    
    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()


def dice_coef(pred, target, smooth=1e-6):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=(2,3))
    denom = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * intersection + smooth) / (denom + smooth)
    return dice.mean()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        dice = dice_coef(pred, target, smooth=self.smooth)
        return 1.0 - dice

def combined_loss(seg_logits, masks, class_logits, class_labels, bce_weight=0.6, dice_weight=0.4, cls_weight=0.3):
    # Segmentation loss
    bce_loss = nn.BCEWithLogitsLoss()
    bce = bce_loss(seg_logits, masks)
    probs = torch.sigmoid(seg_logits)
    dice = DiceLoss()(probs, masks)
    seg_loss = bce_weight * bce + dice_weight * dice
    
    # Classification loss
    cls_loss = nn.CrossEntropyLoss()(class_logits, class_labels)
    
    # Combined loss
    return seg_loss + cls_weight * cls_loss

@torch.no_grad()
def compute_metrics_batch(logits, masks, thresh=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= thresh).float()
    preds_flat = preds.view(-1).cpu().numpy()
    masks_flat = masks.view(-1).cpu().numpy()
    
    unique_preds = np.unique(preds_flat)
    unique_masks = np.unique(masks_flat)
    
    if len(unique_preds) == 1 and len(unique_masks) == 1:
        if unique_preds[0] == unique_masks[0]:
            if unique_preds[0] == 1:
                tp, fp, fn, tn = len(preds_flat), 0, 0, 0
            else:
                tp, fp, fn, tn = 0, 0, 0, len(preds_flat)
        else:
            if unique_preds[0] == 1:
                tp, fp, fn, tn = 0, len(preds_flat), 0, 0
            else:
                tp, fp, fn, tn = 0, 0, len(preds_flat), 0
    else:
        cm = confusion_matrix(masks_flat, preds_flat, labels=[0,1])
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            if cm.shape == (1, 1):
                if unique_masks[0] == 0:
                    tn, fp, fn, tp = cm[0,0], 0, 0, 0
                else:
                    tn, fp, fn, tp = 0, 0, 0, cm[0,0]
            else:
                tn, fp, fn, tp = 0, 0, 0, 0
    
    eps = 1e-8
    iou = tp / (tp + fp + fn + eps)
    iou_bg = tn / (tn + fp + fn + eps)
    miou = (iou + iou_bg) / 2
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    acc = (tp + tn) / (tp + tn + fp + fn + eps)
    
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "iou": float(iou), "miou": float(miou), "dice": float(dice), "precision": float(precision),
            "recall": float(recall), "f1": float(f1), "acc": float(acc)}

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for imgs, masks, class_labels in tqdm(loader, desc="Train batch"):
        imgs = imgs.to(device)
        masks = masks.to(device)
        class_labels = class_labels.to(device)
        optimizer.zero_grad()
        seg_logits, class_logits = model(imgs)
        loss = combined_loss(seg_logits, masks, class_logits, class_labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        
        metrics = compute_metrics_batch(seg_logits, masks)
        running_acc += metrics['acc'] * imgs.size(0)
    
    if scheduler is not None:
        scheduler.step()
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_acc / len(loader.dataset)
    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(model, loader):
    model.eval()
    running_loss = 0.0
    agg = {"tn":0,"fp":0,"fn":0,"tp":0}

    # For classification metrics
    y_true_cls = []
    y_pred_cls = []

    for imgs, masks, class_labels in tqdm(loader, desc="Val batch"):
        imgs = imgs.to(device)
        masks = masks.to(device)
        class_labels = class_labels.to(device)
        out = model(imgs)
        if isinstance(out, (tuple, list)) and len(out) == 2:
            seg_logits, class_logits = out
        else:
            seg_logits = out
            class_logits = None

        loss = combined_loss(seg_logits, masks, class_logits, class_labels)
        running_loss += loss.item() * imgs.size(0)
        metas = compute_metrics_batch(seg_logits, masks)
        for k in ["tn","fp","fn","tp"]:
            agg[k] += metas[k]

        # collect classification preds
        if class_logits is not None:
            preds = class_logits.argmax(dim=1).cpu().numpy()
            y_pred_cls.extend(preds.tolist())
            y_true_cls.extend(class_labels.cpu().numpy().tolist())

    tp, fp, fn, tn = agg["tp"], agg["fp"], agg["fn"], agg["tn"]
    eps = 1e-8
    iou = tp / (tp + fp + fn + eps)
    iou_bg = tn / (tn + fp + fn + eps)
    miou = (iou + iou_bg) / 2
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    acc = (tp + tn) / (tp + tn + fp + fn + eps)

    epoch_loss = running_loss / len(loader.dataset)

    metrics = {"loss": epoch_loss, "iou": iou, "miou": miou, "dice": dice, "precision": precision,
               "recall": recall, "f1": f1, "acc": acc, 
               "confusion": np.array([[tn, fp], [fn, tp]])}

    # Classification metrics if available
    if len(y_true_cls) > 0:
        from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
        labels = list(range(6))
        cm_cls = confusion_matrix(y_true_cls, y_pred_cls, labels=labels)
        per_prec = precision_score(y_true_cls, y_pred_cls, labels=labels, average=None, zero_division=0)
        per_rec = recall_score(y_true_cls, y_pred_cls, labels=labels, average=None, zero_division=0)
        per_f1 = f1_score(y_true_cls, y_pred_cls, labels=labels, average=None, zero_division=0)
        acc_cls = accuracy_score(y_true_cls, y_pred_cls)

        metrics.update({
            'cls_confusion': cm_cls,
            'cls_per_prec': per_prec,
            'cls_per_rec': per_rec,
            'cls_per_f1': per_f1,
            'cls_acc': acc_cls
        })
    else:
        metrics.update({
            'cls_confusion': None,
            'cls_per_prec': None,
            'cls_per_rec': None,
            'cls_per_f1': None,
            'cls_acc': None
        })

    return metrics

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=200, learning_rate=2e-4, patience=10):
    # Only optimize the new heads (decoder + classification head), not the frozen encoder
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=learning_rate)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.7, patience=7, min_lr=1e-6
    )
    
    early_stopping = EarlyStopping(patience=patience, min_delta=1e-4)
    
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    val_ious = []
    val_dices = []
    val_cls_accs = []
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_metrics = validate(model, val_loader)
        
        scheduler.step(val_metrics['loss'])
        
        print(f"Train Loss: {train_loss:.6f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_metrics['loss']:.6f} | Val Acc: {val_metrics['acc']:.4f} | IoU: {val_metrics['iou']:.4f} | Dice: {val_metrics['dice']:.4f} | F1: {val_metrics['f1']:.4f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Print classification metrics if available
        if val_metrics.get('cls_acc') is not None:
            print(f"Val Classification Accuracy: {val_metrics['cls_acc']:.4f}")
            per_prec = val_metrics.get('cls_per_prec')
            per_rec = val_metrics.get('cls_per_rec')
            per_f1 = val_metrics.get('cls_per_f1')
            if per_prec is not None:
                for i, name in enumerate(['MT_Blowhole', 'MT_Break', 'MT_Crack', 'MT_Fray', 'MT_Free', 'MT_Uneven']):
                    print(f"  - {name}: P={per_prec[i]:.3f} R={per_rec[i]:.3f} F1={per_f1[i]:.3f}")
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_metrics['loss'])
        val_accs.append(val_metrics['acc'])
        val_ious.append(val_metrics['iou'])
        val_dices.append(val_metrics['dice'])
        val_cls_accs.append(val_metrics.get('cls_acc'))
        
        if val_metrics['loss'] < best_val_loss - 1e-4:
            best_val_loss = val_metrics['loss']
            torch.save(model.state_dict(), 'best_multitask_mt_resnet50.pth')
            print(f"Saved best model with validation loss: {best_val_loss:.6f}")
        
        if early_stopping(val_metrics['loss'], model):
            print(f'Early stopping triggered after {epoch+1} epochs')
            break
    
    return train_losses, train_accs, val_losses, val_accs, val_ious, val_dices, val_cls_accs

In [ ]:
def main():
    BATCH_SIZE = 6
    LEARNING_RATE = 2e-4
    NUM_EPOCHS = 200
    PATIENCE = 10
    IMG_SIZE = (256, 256)

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    import pathlib, os
    cwd = pathlib.Path.cwd()
    is_kaggle = str(cwd).startswith("/kaggle")

    if is_kaggle:
        # User-provided Kaggle dataset path
        root_dir = "/kaggle/input/datasets/saranjpalani/cvdataset/MagneticTilesDataset_Augmented"
        output_dir = "/kaggle/working"
    else:
        # Local fallback (keep previous notebook behavior)
        augmented_path = cwd / 'MagneticTilesDataset_Augmented'
        root_dir = str(augmented_path) if augmented_path.exists() else str(cwd)
        output_dir = str(cwd)

    print(f"Loading dataset from: {root_dir}")
    print(f"Saving outputs to: {output_dir}")

    train_dataset = MagneticTilesDataset(root_dir, split='train', transform=transform, img_size=IMG_SIZE)
    val_dataset = MagneticTilesDataset(root_dir, split='val', transform=transform, img_size=IMG_SIZE)
    test_dataset = MagneticTilesDataset(root_dir, split='test', transform=transform, img_size=IMG_SIZE)

    num_workers = 2 if is_kaggle else 0
    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")

    model = ResNet50UNet(in_channels=3, num_classes=6, dropout_rate=0.15).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,} (frozen encoder, training decoder + classification head)")

    print("Starting training...")
    train_losses, train_accs, val_losses, val_accs, val_ious, val_dices = train_model(
        model, train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE, PATIENCE
    )

    # Save training curves to output_dir
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    ax1.plot(train_losses, label='Training Loss')
    ax1.plot(val_losses, label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.set_title('Training and Validation Loss')

    ax2.plot(train_accs, label='Training Accuracy')
    ax2.plot(val_accs, label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.set_title('Training and Validation Accuracy')

    ax3.plot(val_ious, label='Validation IoU', color='green')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('IoU')
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.set_title('Validation IoU')

    ax4.plot(val_ious, label='IoU', color='green')
    ax4.plot(val_dices, label='Dice', color='orange')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Score')
    ax4.set_ylim(0, 1)
    ax4.legend()
    ax4.set_title('Validation Metrics')

    plt.tight_layout()
    curves_path = os.path.join(output_dir, 'training_curves_resnet50_multitask_mt.png')
    plt.savefig(curves_path, dpi=300, bbox_inches='tight')
    plt.show()

    # Test evaluation
    best_model_path = os.path.join(output_dir, 'best_multitask_mt_resnet50.pth')
    if os.path.exists(best_model_path):
        print("Loading best model for test evaluation...")
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.to(device)

        print("\nEvaluating on test set...")
        test_metrics = validate(model, test_loader)

        print("\n" + "="*50)
        print("RESNET50 MULTITASK TEST EVALUATION METRICS - MAGNETIC TILES")
        print("="*50)
        print(f"Test set processed with batch_size=1")
        print(f"Loss:            {test_metrics['loss']:.6f}")
        print(f"IoU:             {test_metrics['iou']:.4f}")
        print(f"mIoU:            {test_metrics['miou']:.4f}")
        print(f"Dice Coefficient: {test_metrics['dice']:.4f}")
        print(f"Accuracy:        {test_metrics['acc']:.4f}")
        print(f"Precision:       {test_metrics['precision']:.4f}")
        print(f"Recall:          {test_metrics['recall']:.4f}")
        print(f"F1-Score:        {test_metrics['f1']:.4f}")
        print("Confusion matrix (pixel-level):")
        print(test_metrics["confusion"])
        print("="*50)

        plot_confusion_matrix(test_metrics["confusion"], "Test Set Confusion Matrix (ResNet50 Multitask MT)")

        # --- Classification evaluation (per-image damage-type metrics) ---
        from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

        CLASS_NAMES = ['MT_Blowhole', 'MT_Break', 'MT_Crack', 'MT_Fray', 'MT_Free', 'MT_Uneven']
        y_true = []
        y_pred = []

        model.eval()
        with torch.no_grad():
            for imgs, masks, class_labels in tqdm(test_loader, desc='Classification eval'):
                imgs = imgs.to(device)
                class_labels = class_labels.to(device)
                out = model(imgs)
                if isinstance(out, (tuple, list)) and len(out) == 2:
                    _, class_logits = out
                else:
                    # no classification head available
                    class_logits = None

                if class_logits is None:
                    continue

                probs_batch = torch.softmax(class_logits, dim=1).cpu().numpy()
                preds_batch = probs_batch.argmax(axis=1)

                y_true.extend(class_labels.cpu().numpy().tolist())
                y_pred.extend(preds_batch.tolist())

        if len(y_true) > 0:
            cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
            per_prec = precision_score(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), average=None, zero_division=0)
            per_rec = recall_score(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), average=None, zero_division=0)
            per_f1 = f1_score(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), average=None, zero_division=0)
            acc = accuracy_score(y_true, y_pred)

            print('\nClassification metrics (per-class):')
            for i, name in enumerate(CLASS_NAMES):
                print(f"- {name}: Precision={per_prec[i]:.4f}, Recall={per_rec[i]:.4f}, F1={per_f1[i]:.4f}")
            print(f"Overall classification accuracy: {acc:.4f}")
            print('Confusion matrix (rows=true, cols=pred):')
            print(cm)

            # Save a short report
            report_path = os.path.join(output_dir, 'classification_report_multitask.txt')
            with open(report_path, 'w') as fh:
                fh.write('Per-class metrics (Precision, Recall, F1)\n')
                for i, name in enumerate(CLASS_NAMES):
                    fh.write(f"{name}, {per_prec[i]:.6f}, {per_rec[i]:.6f}, {per_f1[i]:.6f}\n")
                fh.write(f"Overall accuracy, {acc:.6f}\n")
                fh.write('Confusion matrix (rows=true, cols=pred):\n')
                np.savetxt(fh, cm, fmt='%d')

            print(f"Saved classification report to: {report_path}")
        else:
            print('No classification outputs found (model may not have a classification head).')
    else:
        print(f"Best model not found at {best_model_path}. Skipping test evaluation.")

    print("ResNet50 Multi-Task Magnetic Tiles training and evaluation completed!")


if __name__ == "__main__":
    main()


Loading dataset from: c:\Users\Admin\Downloads\cvprojjjj\Segmentation-of-Damaged-Magnetic-Tiles\MagneticTilesDataset_Augmented
Training samples: 2249
Validation samples: 268
Test samples: 271


C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Total parameters: 65,249,863
Trainable parameters: 41,741,831 (frozen encoder, training decoder + classification head)
Starting training...

Epoch 1/200


Val batch: 100%|██████████| 134/134 [01:24<00:00,  1.59it/s]


Train Loss: 0.961215 | Train Acc: 0.9228
Val Loss: 0.738986 | Val Acc: 0.9721 | IoU: 0.0000 | Dice: 0.0000 | F1: 0.0000
Learning Rate: 2.00e-04
Saved best model with validation loss: 0.738986

Epoch 2/200


Train batch:   1%|▏         | 5/375 [00:10<12:27,  2.02s/it]


KeyboardInterrupt: 